In [14]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!pip install -q kaggle
!kaggle competitions download -c store-sales-time-series-forecasting

100% 21.4M/21.4M [00:00<00:00, 122MB/s] 



In [15]:
!unzip -q store-sales-time-series-forecasting.zip -d store_sales_data

In [16]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_squared_log_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

In [ ]:
!ls store_sales_data

ls: cannot access 'store_sales_data': No such file or directory


In [17]:
train_df = pd.read_csv("store_sales_data/train.csv")
test_df = pd.read_csv("store_sales_data/test.csv")
stores_df = pd.read_csv("store_sales_data/stores.csv")
oil_df = pd.read_csv("store_sales_data/oil.csv")
holidays_df = pd.read_csv("store_sales_data/holidays_events.csv")

In [21]:
for df in [train_df, test_df, oil_df, holidays_df]:
    df['date'] = pd.to_datetime(df['date'])

In [22]:
oil_df['dcoilwtico'] = oil_df['dcoilwtico'].ffill().bfill()

In [23]:
def merge_datasets(df):
    df = df.merge(stores_df, on='store_nbr', how='left')
    df = df.merge(oil_df, on='date', how='left')
    df['dcoilwtico'] = df['dcoilwtico'].ffill().bfill()
    return df

In [24]:
train_df= merge_datasets(train_df)
test_df = merge_datasets(test_df)

In [25]:
train_df.columns

Index(['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city',
       'state', 'type', 'cluster', 'dcoilwtico'],
      dtype='object')

In [26]:
df_full = pd.concat([train_df , test_df] , ignore_index= True)

In [27]:
df_full = df_full.sort_values(['store_nbr' , 'date' , 'family']).reset_index(drop = True)

In [28]:
df_full['year'] = df_full['date'].dt.year
df_full['month'] = df_full['date'].dt.month
df_full['day'] = df_full['date'].dt.day
df_full['dayofweek'] = df_full['date'].dt.dayofweek
df_full['weekend'] = df_full['dayofweek'].isin([5 , 6 ]).astype(int)

In [29]:
for lag in [1 ,7 ,14 ] :
  df_full[f'sales_lag_{lag}'] = df_full.groupby(['store_nbr' , 'family'])['sales'].shift(lag)

df_full['sales_roll_mean'] = df_full.groupby(['store_nbr' , 'family'])['sales'].transform(lambda x : x.shift(1).rolling(lag).mean())
df_full['sales_roll_std'] = df_full.groupby(['store_nbr' , 'family'])['sales'].transform(lambda x : x.shift(1).rolling(lag).std())


categorical_cols = ['family' , 'city' , 'state' , 'type']
df_full = pd.get_dummies(df_full , columns= categorical_cols , drop_first= True)


In [30]:
train_processed = df_full[df_full['sales'].notna()].copy()
test_processed = df_full[df_full['sales'].isna()].copy()

In [31]:
train_processed = train_processed.fillna(0)
test_processed = test_processed.fillna(0)

In [32]:
train_processed = train_processed[train_processed['date'] >= '2016-01-01']
train_mask = train_processed['date'] < '2017-07-15'
val_mask = train_processed['date'] >= '2017-07-15'

In [33]:
drop_cols = ['id' , 'date' , 'sales' ]
x = train_processed.drop(drop_cols , axis = 1)
y = train_processed['sales']

In [34]:
x_train , y_train = x[train_mask] , y[train_mask]
x_val , y_val = x[val_mask] , y[val_mask]

In [35]:
models = {
    'Hist Gradient Boosting': HistGradientBoostingRegressor(
        max_iter=100,
        learning_rate=0.08,
        random_state=42
    ),
    'Random Forest Regressor': RandomForestRegressor(
        n_estimators= 30 ,
        max_depth = 12 ,
        random_state = 42 ,
        n_jobs = -1
    ),
    'XGBoost Regression ': xgb.XGBRegressor(
        n_estimators = 80 ,
        learning_rate = 0.08 ,
        max_depth = 6 ,
        random_state = 42 ,
        tree_method = 'hist' ,
        n_jobs = -1
    ),
    'Ridge Regression': Ridge(alpha=1.0)
}

In [36]:
benchmark_results = []
sample_size = 200000

for name, model in models.items():
    print(f'training {name}...')

    if name in ['Random Forest Regressor', 'XGBoost Regression '] and len(x_train) > sample_size:
        x_fit = x_train.sample(sample_size, random_state=42)
        y_fit = y_train.loc[x_fit.index]
    else:
        x_fit, y_fit = x_train, y_train

    model.fit(x_fit, y_fit)

    train_pred = np.clip(model.predict(x_fit), 0, None)
    train_r2 = r2_score(y_fit, train_pred)

    val_pred = np.clip(model.predict(x_val), 0, None)
    val_r2 = r2_score(y_val, val_pred)

    rmsle = np.sqrt(mean_squared_log_error(y_val, val_pred))

    benchmark_results.append({
        'Model': name,
        'Train R² Score': f"{round(train_r2 * 100, 2)}%",
        'Test R² Score': f"{round(val_r2 * 100, 2)}%",
        'RMSLE (Kaggle Metric)': round(rmsle, 4),
    })

print(pd.DataFrame(benchmark_results).to_string(index=False))

training Hist Gradient Boosting...
training Random Forest Regressor...
training XGBoost Regression ...
training Ridge Regression...
                  Model Train R² Score Test R² Score  RMSLE (Kaggle Metric)
 Hist Gradient Boosting         95.05%        96.97%                 0.8318
Random Forest Regressor         97.22%        96.73%                 0.4156
    XGBoost Regression          97.41%        96.92%                 0.7409
       Ridge Regression         91.08%         96.4%                 1.4023


In [37]:
import joblib

best_model = RandomForestRegressor(
    n_estimators=30,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

best_model.fit(x_train, y_train)

joblib.dump(best_model, "best_model.pkl")

['best_model.pkl']

In [38]:
joblib.dump(list(x_train.columns), "feature_columns.pkl")

print("Feature names saved successfully.")

Feature names saved successfully.


In [39]:
import os

print(os.listdir())

['.config', 'store_sales_data', 'feature_columns.pkl', 'store-sales-time-series-forecasting.zip', 'best_model.pkl', 'kaggle.json', 'sample_data']
